# D19 S1 A1 - ResNet Residual Blocks and Batch Normalization

NumPy implementation of ReLU, batch normalization, identity residual blocks, projection shortcuts, and width-changing residual blocks.


In [ ]:
import numpy as np

rng = np.random.RandomState(0)
x = rng.randn(16, 6)
W = rng.randn(6, 6) * 0.1
gamma = np.ones(6)
beta = np.zeros(6)


In [ ]:
def relu(x):
    return np.maximum(0, x)


def batch_norm(x, gamma, beta, eps=1e-5):
    mean = np.mean(x, axis=0)
    var = np.var(x, axis=0, ddof=0)
    xhat = (x - mean) / np.sqrt(var + eps)
    return gamma * xhat + beta


def residual_block(x, W, gamma, beta):
    F = batch_norm(x @ W, gamma, beta)
    return relu(F + x)


def projection_shortcut(x, Wp):
    return x @ Wp


def residual_block_proj(x, W, Wp, gamma, beta):
    F = batch_norm(x @ W, gamma, beta)
    shortcut = projection_shortcut(x, Wp)
    return relu(F + shortcut)


## Verification


In [ ]:
rng = np.random.RandomState(0)
x = rng.randn(200, 4) * 5.0 + 2.0
y = batch_norm(x, np.ones(4), np.zeros(4))
print(np.round(y.mean(0), 6), np.round(y.var(0), 4))

x = rng.randn(20, 5)
W = rng.randn(5, 5)
out = residual_block(x, W, np.zeros(5), np.zeros(5))
print(np.allclose(out, np.maximum(0, x)))

print(projection_shortcut(np.ones((2, 3)), np.ones((3, 4))).shape)


## Additional checks matching the autograder requirements


In [ ]:
# 1. ReLU
a = np.array([-2.0, 0.0, 3.0])
assert np.array_equal(relu(a), np.array([0.0, 0.0, 3.0]))

# 2-4. Batch norm: zero mean, unit variance, scale + shift
rng = np.random.RandomState(1)
bn_x = rng.randn(500, 4) * 3 + 7
z = batch_norm(bn_x, np.ones(4), np.zeros(4))
assert np.allclose(z.mean(axis=0), 0, atol=1e-6)
assert np.allclose(z.var(axis=0), 1, atol=1e-4)
scaled = batch_norm(bn_x, np.ones(4) * 2, np.ones(4) * 3)
assert np.allclose(scaled.mean(axis=0), 3, atol=1e-6)
assert np.allclose(scaled.std(axis=0), 2, atol=1e-4)

# 5 and 7. Identity residual block shape and skip path
x_id = rng.randn(20, 5)
W_id = rng.randn(5, 5)
out_id = residual_block(x_id, W_id, np.zeros(5), np.zeros(5))
assert out_id.shape == x_id.shape
assert np.allclose(out_id, relu(x_id))

# 6 and 8. Projection shortcut and projected residual block
xp = rng.randn(10, 3)
Wp = rng.randn(3, 4)
Wproj = rng.randn(3, 4)
shortcut = projection_shortcut(xp, Wp)
assert shortcut.shape == (10, 4)
assert np.allclose(shortcut, xp @ Wp)

out_proj = residual_block_proj(xp, Wproj, Wp, np.zeros(4), np.zeros(4))
assert out_proj.shape == (10, 4)
assert np.allclose(out_proj, relu(xp @ Wp))

print('All checks passed.')
